In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

In [2]:
load_dotenv()

NOTEBOOK_DIR = Path.cwd()
dotenv_path = NOTEBOOK_DIR / '.env'
load_dotenv(dotenv_path=str(dotenv_path))

GIGACHAT_API_KEY = os.environ.get("GIGACHAT_API_KEY")

In [10]:
from langchain_gigachat import GigaChat

llm_gigachat = GigaChat(
    model="GigaChat:latest",
    scope="GIGACHAT_API_B2B",
    credentials=GIGACHAT_API_KEY,
    verify_ssl_certs=False,
)

In [4]:
import json
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from pydantic import Field

ORDERS_STATUSES_DATA = {
    "a42": "Доставляется",
    "b61": "Выполнен",
    "k37": "Отменен",
}


@tool
def get_order_status(order_id: str = Field(description="Identifier of order")) -> str:
    """Get status of order by order identifier"""
    return ORDERS_STATUSES_DATA.get(order_id, f"Не существует заказа")


def cancel_order(order_id: str) -> str:
    if order_id not in ORDERS_STATUSES_DATA:
        return f"Не существует заказа с order_id={order_id}"
    if ORDERS_STATUSES_DATA[order_id] != "Отменен":
        ORDERS_STATUSES_DATA[order_id] = "Отменен"
        return "Заказ успешно отменен"
    return "Заказ уже отменен"

In [5]:
print("Name:", get_order_status.name)
print("Description:", get_order_status.description)
print("Arguments:", get_order_status.args) 

Name: get_order_status
Description: Get status of order by order identifier
Arguments: {'order_id': {'description': 'Identifier of order', 'title': 'Order Id', 'type': 'string'}}


In [6]:
print(json.dumps(get_order_status.args_schema.schema(), indent=4))

{
    "description": "Get status of order by order identifier",
    "properties": {
        "order_id": {
            "description": "Identifier of order",
            "title": "Order Id",
            "type": "string"
        }
    },
    "required": [
        "order_id"
    ],
    "title": "get_order_status",
    "type": "object"
}


C:\Users\Tumbi\AppData\Local\Temp\ipykernel_7716\822697386.py:1: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(json.dumps(get_order_status.args_schema.schema(), indent=4))


In [7]:
print("Result:", get_order_status.invoke({"order_id": "a42"}))

Result: Доставляется


# Использование Tool Calling с LLM

In [8]:
llm_with_tools = llm.bind_tools([get_order_status])
ai_message = llm_with_tools.invoke("What about my order with id equal to k37?")

In [9]:
for tool_call in ai_message.tool_calls:
    if tool_call["name"] == get_order_status.name:
        tool_message = get_order_status.invoke(tool_call)

In [10]:
ai_message.tool_calls

[{'name': 'get_order_status',
  'args': {'order_id': 'k37'},
  'id': 'e38c0ac3-f9a4-4259-b82c-17a765fedd19',
  'type': 'tool_call'}]

In [11]:
tool_message

ToolMessage(content='Отменен', name='get_order_status', tool_call_id='e38c0ac3-f9a4-4259-b82c-17a765fedd19')

In [12]:
result = llm_with_tools.invoke([
    HumanMessage("What about my order with id equal to k37?"),
    ai_message,
    tool_message
])

result.content

'Ваш заказ с идентификатором `k37` отменён.'

# LangChain Agents

In [ ]:
import time
from langchain_core.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


tools = [get_order_status, cancel_order]
llm = GigaChat(
    model="GigaChat:latest",
    scope="GIGACHAT_API_B2B",
    credentials=GIGACHAT_API_KEY,
    verify_ssl_certs=False,
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", (
            "Твоя задача отвечать на вопросы клиентов об их заказах. Отвечай пользователю подробно и вежливо."
        )),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\main\data_science\projects\dl_practice\.venv\Lib\site-packages\langchain\agents\__init__.py)

In [ ]:
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
result = agent_executor.invoke({"input": "Отмени заказ k37"})
print(result)

# Существующие тулы

In [ ]:
from langchain.agents import Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(lang="en"))
wikipedia_tool = Tool(
    name="wikipedia",
    description="Search in Wikipedia knowledge database.",
    func=wikipedia.run,
)
result = wikipedia_tool.invoke("Large Language Models")
print(result)

# Существующие тулы

In [16]:
from langchain_core.tools import Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(lang="en"))
wikipedia_tool = Tool(
    name="wikipedia",
    description="Search in Wikipedia knowledge database.",
    func=wikipedia.run,
)
result = wikipedia_tool.invoke("Large Language Models")
print(result)

Page: Large language model
Summary: A large language model (LLM) is a language model trained with self-supervised machine learning on a vast amount of text, designed for natural language processing tasks, especially language generation. The largest and most capable LLMs are generative pre-trained transformers (GPTs) and provide the core capabilities of modern chatbots. LLMs can be fine-tuned for specific tasks or guided by prompt engineering. These models acquire predictive power regarding syntax, semantics, and ontologies inherent in human language corpora, but they also inherit inaccuracies and biases present in the data they are trained on.
They consist of billions to trillions of parameters and operate as general-purpose sequence models, generating, summarizing, translating, and reasoning over text. LLMs represent a significant new technology in their ability to generalize across tasks with minimal task-specific supervision, enabling capabilities like conversational agents, code ge

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.tools import Tool
from langchain_experimental.utilities import PythonREPL


class ToolInput(BaseModel):
    code: str = Field(description="Python code to execute.")


python_repl = PythonREPL()
repl_tool = Tool(
    name="python_repl",
    description="Executes python code and returns the result. The code runs in a static sandbox without interactive mode, so print output or save output to a file.",
    func=python_repl.run,
)
repl_tool.args_schema = ToolInput

result = repl_tool.invoke("print(1+1)")
print(result)

Python REPL can execute arbitrary code. Use with caution.


2



In [32]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent  
from langchain_core.tools import StructuredTool
from langchain_experimental.utilities import PythonREPL


class PythonInput(BaseModel):
    code: str = Field(description="Python код для выполнения")


python_repl = PythonREPL()
python_tool = StructuredTool.from_function(
    func=lambda code: python_repl.run(code),
    name="PythonREPL",
    description="Выполняет Python код в безопасной среде",
    args_schema=PythonInput,
)

agent = create_agent(llm, tools=[python_tool])

result = agent.invoke({"messages": [("human", "Вычисли NPV проекта: cash flows = [1000, 2000, 3000], discount_rate=0.1")]})
result['messages'][-1].content

'Задача решена корректно.\n\n**Итоговый результат:**  \nNPV данного инвестиционного проекта составляет примерно $4\\,815.93.\n\nТаким образом, проект является прибыльным, так как NPV положителен.'

# ReAct Agent

In [36]:
import time
import datetime
from uuid import uuid4
from typing import Optional, Annotated

from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.agents import create_agent
from langsmith import Client

In [ ]:
@tool(name_or_callable="current-year-tool")
def get_this_year_tool() -> Annotated[int, "Current year"]:
    """Get the current year"""
    time.sleep(1)
    return datetime.datetime.now().year


class WikiInputs(BaseModel):
    """Inputs to the wikipedia tool."""

    query: str = Field(
        description="query to look up in Wikipedia"
    )


wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(lang="ru"))
wikipedia_tool = Tool(
    name="wikipedia-tool",
    description="Look up things in Wikipedia",
    args_schema=WikiInputs,
    func=wikipedia.run,
)
TOOLS = [wikipedia_tool, get_this_year_tool]

In [42]:
client = Client()
prompt = client.pull_prompt("sanchezzz/russian_react_chat").template

print(prompt)

# prompt = ChatPromptTemplate.from_messages([
#     ("system", """Ты полезный ассистент. 
#     Используй инструменты для поиска фактов.
#     Отвечай КРАТКО на русском языке."""),
#     ("human", "{input}"),
#     MessagesPlaceholder(variable_name="agent_scratchpad")
# ])

# print(prompt)

Assistant is a large language model.

Assistant is designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics. Espessialy, assistant is usefull with movie questions. As a language model, Assistant is able to generate human-like text based on the input it receives, allowing it to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand. The creator of assistant is Alexander Posobilo.

Assistant is constantly learning and improving, and its capabilities are constantly evolving. It is able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. Additionally, Assistant is able to generate its own text based on the input it receives, allowing it to engage in discussions and provide explanations and descriptions on a wide range of top

In [46]:
agent = create_agent(llm, TOOLS, system_prompt=prompt)

result = agent.invoke(
    {
        "input": "Сколько лет прошло с появления передачи Поле чудес в эфире? Кто её ведущий сегодня?",
        "chat_history": []
    }
)
print(result)

NameError: name 'uuid' is not defined

In [ ]:
from typing import List, Dict, Any, Optional, TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from pydantic import BaseModel, Field
import datetime
import time
import logging
from langsmith import traceable


# Определение инструментов
@tool
def get_this_year_tool() -> int:
    """Get the current year"""
    time.sleep(1)
    return datetime.datetime.now().year

class WikiInputs(BaseModel):
    """Inputs to the wikipedia tool."""
    query: str = Field(description="query to look up in Wikipedia")

# Создание инструментов
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(lang="ru"))

# Определение состояния
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


def create_agent_graph(llm):
    try:
        # Создание промпта
        prompt = ChatPromptTemplate.from_messages([
            ("system", """Ты полезный ассистент.
            Используй инструменты для поиска фактов.
            Отвечай КРАТКО на русском языке."""),
            MessagesPlaceholder(variable_name="messages"),
            ("user", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])

        # Определение инструментов
        tools = [wikipedia, get_this_year_tool]

        # Создание узла модели
        def call_model(state: AgentState):
            messages = state["messages"]
            response = llm.invoke(messages)
            return {"messages": [response]}

        # Создание графа
        workflow = StateGraph(AgentState)

        # Добавление узлов
        workflow.add_node("model", call_model)
        workflow.add_node("tools", ToolNode(tools))

        # Определение рёбер
        workflow.add_edge(START, "model")

        # Условие для вызова инструментов
        def should_use_tools(state: AgentState):
            last_message = state["messages"][-1]
            if hasattr(last_message, "tool_calls"):
                return "tools"
            return END

        workflow.add_conditional_edges("model", should_use_tools)

        # Возврат к модели после использования инструментов
        workflow.add_edge("tools", "model")

        # Компиляция графа
        app = workflow.compile()

        return app
    except Exception as e:
        raise


def run_agent(app, question: str):
    try:
        # Создание начального состояния
        initial_state = {
            "messages": [
                HumanMessage(content=question)
            ]
        }

        # Выполнение
        result = app.invoke(initial_state)

        # Возвращаем последний ответ
        return result["messages"][-1].content
    except Exception as e:
        raise


app = create_agent_graph(llm)

# Запуск
question = "Сколько лет прошло с появления передачи Поле чудес в эфире? Кто её ведущий сегодня?"
result = run_agent(app, question)
print(result)


INFO:httpx:HTTP Request: POST https://gigachat.devices.sberbank.ru/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://gigachat.devices.sberbank.ru/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://gigachat.devices.sberbank.ru/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://gigachat.devices.sberbank.ru/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://gigachat.devices.sberbank.ru/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://gigachat.devices.sberbank.ru/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://gigachat.devices.sberbank.ru/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://gigachat.devices.sberbank.ru/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://gigachat.devices.sberbank.ru/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Req

ReadTimeout: The read operation timed out

# LangGraph

In [11]:
import random
from typing import Literal
from typing_extensions import TypedDict

from langchain_core.output_parsers import StrOutputParser


class State(TypedDict):
    query: str
    resolver: str
    answer: str


def choice_resolver(state: State) -> State:
    resolver = "support" if random.random() > 0.5 else "llm"
    state["resolver"] = resolver
    return state


def send_to_support(state: State) -> State:
    print(f"New message for Support: {state['query']}")
    return state


def llm(state: State) -> State:
    messages = [
        ("system", "You are a friendly chatbot. Your task is answer the question as short as possible"),
        ("human", "{question}"),
    ]
    prompt = ChatPromptTemplate(messages)
    
    chain = prompt | llm_gigachat | StrOutputParser()
    answer = chain.invoke({"question": state["query"]})
    state["answer"] = answer
    return state


def send_to_user(state: State) -> State:
    print(f"New message for User: {state['answer']}")
    return state


def route_by_resolver(state: State) -> Literal["send_to_support", "llm"]:
    if state["resolver"] == "support":
        return "send_to_support"
    else:
        return "llm"
    

builder = StateGraph(State)
builder.add_node("choice_resolver", choice_resolver)
builder.add_node("send_to_support", send_to_support)
builder.add_node("llm", llm)
builder.add_node("send_to_user", send_to_user)

builder.add_edge(START, "choice_resolver")
builder.add_conditional_edges("choice_resolver", route_by_resolver)
builder.add_edge("send_to_support", END)
builder.add_edge("llm", "send_to_user")
builder.add_edge("send_to_user", END)

graph = builder.compile()

In [12]:
with open("graph.png", "wb") as f:
    f.write(graph.get_graph().draw_mermaid_png())

In [13]:
result = graph.invoke({"query" : "Hi, my computer is not working!"})
print(result)

INFO:httpx:HTTP Request: POST https://ngw.devices.sberbank.ru:9443/api/v2/oauth "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://gigachat.devices.sberbank.ru/api/v1/chat/completions "HTTP/1.1 200 OK"


New message for User: Oh no! What seems to be the problem?
{'query': 'Hi, my computer is not working!', 'resolver': 'llm', 'answer': 'Oh no! What seems to be the problem?'}


# LangGraph: агенты

In [15]:
from typing import Sequence
from langgraph.graph.message import add_messages
from langchain_core.runnables import RunnableConfig


class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    number_of_steps: int


@tool(return_direct=True)
def get_this_year_tool() -> int:
    """Получить текущий год"""
    return datetime.datetime.now().year


class WikiInput(BaseModel):
    query: str = Field(
        description="Запрос для поиска в Википедия"
    )

wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(lang="ru"))


@tool(return_direct=True, args_schema=WikiInput)
def search_using_wikipedia(query: str) -> str:
    """Позволяет искать что-то в Википедия"""
    return wikipedia.run({"query": query})


tools = [search_using_wikipedia, get_this_year_tool]
tools_by_name = {tool.name: tool for tool in tools}


def call_model(
    state: AgentState,
    config: RunnableConfig,
):

    model = llm_gigachat.bind_tools(tools)
    response = model.invoke(state["messages"], config)
    return {"messages": [response], "number_of_steps": state["number_of_steps"] + 1}


def call_tool(state: AgentState):
    outputs = []
    for tool_call in state["messages"][-1].tool_calls:
        tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
        outputs.append(
            ToolMessage(
                content=tool_result,
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )
    return {"messages": outputs, "number_of_steps": state["number_of_steps"] + 1}


def should_continue(state: AgentState) -> str:
    messages = state["messages"]
    if not messages[-1].tool_calls:
        return "end"
    return "continue"


builder = StateGraph(AgentState)

builder.add_node("llm", call_model)
builder.add_node("tools",  call_tool)

builder.add_edge(START, "llm")
builder.add_conditional_edges(
    "llm",
    should_continue,
    {
        "continue": "tools",
        "end": END,
    }
)
builder.add_edge("tools", "llm")
graph = builder.compile()


In [16]:
inputs = {"messages": [("user", "Сколько лет прошло с появления передачи Поле чудес в эфире? Кто её ведущий сегодня?")], "number_of_steps": 0}
state = graph.invoke(inputs)

for message in state["messages"]:
    message.pretty_print()
    print("=" * 80 + "\n\n")

INFO:httpx:HTTP Request: POST https://gigachat.devices.sberbank.ru/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://gigachat.devices.sberbank.ru/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://gigachat.devices.sberbank.ru/api/v1/chat/completions "HTTP/1.1 200 OK"


================================ Human Message =================================

Сколько лет прошло с появления передачи Поле чудес в эфире? Кто её ведущий сегодня?


================================== Ai Message ==================================
Tool Calls:
  get_this_year_tool (0e4f4457-1168-489b-86f0-9a2166cc7e85)
 Call ID: 0e4f4457-1168-489b-86f0-9a2166cc7e85
  Args:


================================= Tool Message =================================
Name: get_this_year_tool

2026


================================== Ai Message ==================================
Tool Calls:
  search_using_wikipedia (e2dfe316-760a-4d98-bd13-6fa63e4bd590)
 Call ID: e2dfe316-760a-4d98-bd13-6fa63e4bd590
  Args:
    query: Поле чудес дата начала трансляции


================================= Tool Message =================================
Name: search_using_wikipedia

Page: Поле чудес
Summary: Капитал-шоу «По́ле чуде́с» — советская и российская телеигра, выходящая каждую пятницу в 19:45 на ОРТ/«Первом ка